# 1 Module, Globals und Daten

## 1.1 Module

In [ ]:
##### IMPORTS #####

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

import sklearn.metrics as skm
from sklearn.feature_selection import RFE, RFECV
from sklearn.model_selection import train_test_split, StratifiedKFold

import matplotlib.pyplot as plt

import os

## 1.2 Globals

In [ ]:
##### Globals #####
dataPath = 'datasets/harus/'


## 1.3 Daten

In [ ]:
xdata = pd.read_csv(dataPath + "xdata.csv", sep=";")                     
ydata = pd.read_csv(dataPath + "ydata.csv", sep=";")

xtrain_old, xvaltest, ytrain, yvaltest = train_test_split( 
    xdata,
    ydata,
    random_state=0,
    train_size=0.66,
    stratify=ydata                                  # Preserve label imbalance across train- and test datasets
)

xval_old, xtest_old, yval, ytest = train_test_split( 
    xvaltest,
    yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=yvaltest                                  # Preserve label imbalance across train- and test datasets
)

ev_val_old = [(xval_old,yval)]
ev_all_old = [(xtrain_old,ytrain),(xval_old,yval),(xtest_old,ytest)]

## 1.4 Modellimport

In [ ]:
clf = XGBClassifier(objective = "multi:softmax",
                    tree_method = 'exact',
                    n_estimators = 194,
                    max_depth = 4)
clf.load_model("model-exports/aktuelle-exports/base_harus.json")

# 2 Feature Selection

## 2.1 Embedded: Feature Importance

### 2.1.0 Define related funcs

In [ ]:
def getFeatureImportances(model: XGBClassifier, prints: bool = False) -> list:
    
    # Erstellt eine Liste mit den relativen Wichtigkeiten aller Features des Modells nach durchschnittlichem Gain.

    featurenames  = [str(i) for i in list(model.feature_names_in_)]
    importances = [float(i) for i in list(model.feature_importances_)]

    namedimportances = list(zip(importances, featurenames))

    namedimportances = sorted(namedimportances, key=lambda x: x[0], reverse=True)

    if prints == True:
        i = 1
        for feature in namedimportances:
            print(f'{i}. {feature[1]}: \t{feature[0]}')
            i += 1

    return(namedimportances)

def getMostImportantFeatures(featureImportances, minGain: float = 0.95):
    
    # Gibt die ersten Elemente einer Liste entgegen, bis ein Wert minGain ueberschritten worden ist

    localRelWeights = []
    localFeatureNames = []
    weightsum = 0

    for feature in featureImportances:
        if (weightsum) < minGain:
            localRelWeights.append(feature[0])
            localFeatureNames.append(feature[1])
            
            weightsum += feature[0]

    return localFeatureNames, localRelWeights

def getBestDepth(listOfAccuracies, listOfSizes):  
        data = {'Accuracy': listOfAccuracies,
                'Size': listOfSizes}
        AccVsSize = pd.DataFrame(data)
        Suitable = AccVsSize[AccVsSize['Size']<1024]
        bestAccsWithinBoundaries = Suitable[Suitable['Accuracy']==max(Suitable['Accuracy'])]
        bestDepth = bestAccsWithinBoundaries[bestAccsWithinBoundaries['Size'] == min(bestAccsWithinBoundaries['Size'])]

        return bestDepth

### 2.1.1 Get (sorted) Feature Importances

In [ ]:
ranking = getFeatureImportances(clf, False)
relWeights = []
featureNames = []

for feature in ranking:
    relWeights.append(feature[0])
    featureNames.append(feature[1])

rwlist = [
    relWeights[0:50],
    relWeights[50:100],
    relWeights[100:150],
    relWeights[150:200],
    relWeights[200:250],
    relWeights[250:300],
    relWeights[300:350],
    relWeights[350:400],
    relWeights[400:450],
    relWeights[450:500],
    relWeights[500:550],
    relWeights[550:561]
]
fnlist = [
    featureNames[0:50],
    featureNames[50:100],
    featureNames[100:150],
    featureNames[150:200],
    featureNames[200:250],
    featureNames[250:300],
    featureNames[300:350],
    featureNames[350:400],
    featureNames[400:450],
    featureNames[450:500],
    featureNames[500:550],
    featureNames[550:561]
]

### 2.1.2 Plot Top 10 Features

In [ ]:
fig, ax = plt.subplots()

y_pos = np.arange(10)

ax.set_ylim(-1, 10)
ax.barh(y_pos, rw0[:10], align='center')
ax.set_yticks(y_pos, labels=fn0[:10])
ax.invert_yaxis()  # labels read top-to-bottom
ax.set_xlabel('Informationsgewinn (Gain)')
ax.set_title('HARUS: Relative Wichtigkeiten der Merkmale (Top 10)')

index = 0
for i in rw0[:10]:
    if i < 0.03:
        plt.text(i,index + 0.15,str(round(i*100,4))+"%")
        index +=1
    else:
        plt.text(0.0025,index + 0.15,str(round(i*100,4))+"%")
        index +=1

plt.show()

### 2.1.3 Plot all Feature Importances (50 each)

In [ ]:
def plotImportances(weights, names, number):
    fig, ax = plt.subplots()

    if len(weights) == 50: 
        fig.set_figheight(15)

    y_pos = np.arange(len(names))

    ax.set_ylim(-1, len(names))
    ax.barh(y_pos, weights, align='center')
    ax.set_yticks(y_pos, labels=names)
    ax.invert_yaxis() 
    ax.set_xlabel('Informationsgewinn (Gain)')
    ax.set_title('HARUS: Relative Wichtigkeiten der Merkmale [' + str(number*50) + ":" + str((number*50)+50) + "]")

    index = 0
    for i in weights:
        if i < (weights[0]/7):
            plt.text(i,index + 0.15,str(round(i*100,4))+"%")
            index +=1
        else:
            plt.text((weights[0]/100),index + 0.15,str(round(i*100,4))+"%")
            index +=1

    plt.show()

for i in range(0, 12):
    plotImportances(rwlist[i], fnlist[i], i)

### 2.1.4 Get most Important Features

In [ ]:
mINames, mIWeights = getMostImportantFeatures(ranking)
print(f"Gesamte Anzahl der Merkmale: {len(ranking)}")
print(f"Anzahl der Merkmale mit 95% Informationsgewinn: {len(mINames)}")

### 2.1.4 Teilmenge erstellen

In [ ]:
xtrain = xtrain_old[mINames]
xval = xval_old[mINames]
xtest = xtest_old[mINames]

ev_val = [(xval,yval)]
ev_all = [(xtrain,ytrain),(xval,yval),(xtest,ytest)]

### 2.1.5 Modell trainieren

In [ ]:
clf_fs = XGBClassifier(
    objective = "multi:softmax",
    tree_method = 'exact',
    n_estimators = 194,
    max_depth = 4
)

clf_fs.fit(
    xtrain,ytrain,
    eval_set=ev_val,
    verbose = 0
)

yhat = clf_fs.predict(xtest)

print(f'Accuracy-Score: \t{skm.accuracy_score(ytest, yhat)}') 

### 2.1.6 n_estimators und max_depth ermitteln

In [ ]:
bestIterList = []
accList = []
jsonSizeList = []

maximum = 30

for i in range(1, maximum+1):

    clf_temp = XGBClassifier(
        objective = "multi:softmax",
        tree_method = "exact",
        n_estimators = 100000,
        early_stopping_rounds = 10,
        max_depth = i
    )
    clf_temp.fit(
        xtrain, ytrain,
        eval_set = ev_val,
        verbose = 0
    )
    
    bestIter_temp = clf_temp.best_iteration

    clf_temp_bestIter = XGBClassifier(
        objective = "multi:softmax",
        tree_method = "exact",
        n_estimators = bestIter_temp,
        max_depth = i
    )
    clf_temp_bestIter.fit(xtrain, ytrain)

    yhat_temp_bestIter = clf_temp_bestIter.predict(xtest)

    acc_temp = skm.accuracy_score(ytest, yhat_temp_bestIter)
    
    clf_temp_bestIter.save_model("clf_temp.json")

    jsonSizeList.append(os.path.getsize('clf_temp.json'))
    accList.append(acc_temp)
    bestIterList.append(bestIter_temp)

    os.remove('clf_temp.json')

### 2.1.7 Graphen für Suche nach Params plotten

In [ ]:
jsonSizeList_KB = [x/1024 for x in jsonSizeList]
bestIterationValues = getBestDepth(accList, jsonSizeList_KB)
best_depth = bestIterationValues.index[0]+1

In [ ]:
fig, ax = plt.subplots(1,1)

ax.set_xlabel("Maximale Baumtiefe “max_depth”")
ax.set_ylabel("Balancierte Genauigkeit")
ax.tick_params(axis='y', labelcolor="blue")
ax.plot(range(1,31),accList[:30],color="blue")
ax.set_xlim(0,30)

twin = ax.twinx()

twin.set_ylabel("Modellgröße in Kilobytes")
# twin.set_ylim(200,1150)
twin.tick_params(axis='y', labelcolor="red")
twin.plot(range(1,31),jsonSizeList_KB[:30],color="red")

twin.axhline(y=1024, color="red", linestyle=":", label="Maximale Modellgröße (1024kB)")
twin.axvline(x=best_depth, color="black", linestyle=":", label="Bester Wert “max_depth” unter 1024kB")

twin.legend(loc='upper right')

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1)

ax.set_xlabel("Maximale Baumtiefe “max_depth”")
ax.set_ylabel("Balancierte Genauigkeit")
ax.tick_params(axis='y', labelcolor="blue")
ax.plot(range(5,11),accList[4:10],color="blue")
# ax.set_xticks([1,2,3,4,5])
# ax.set_xlim((0.8),5)

twin = ax.twinx()

twin.set_ylabel("Modellgröße in Kilobytes")
twin.set_ylim(200,1150)
twin.tick_params(axis='y', labelcolor="red")
twin.plot(range(5,11),jsonSizeList_KB[4:10],color="red")

twin.axhline(y=1024, color="red", linestyle=":", label="Maximale Modellgröße (1024kB)")
twin.axvline(x=best_depth, color="black", linestyle=":", label="Bester Wert “max_depth” unter 1024kB")

twin.legend(loc='upper right')

fig.tight_layout()
plt.show()

### 2.1.8 Finales Modell erzeugen

In [ ]:
clf_final = XGBClassifier(
    objective = "multi:softmax",
    tree_method = 'exact',
    n_estimators = bestIterList[best_depth-1],
    max_depth = best_depth
)
clf_final.fit(
    xtrain,ytrain,
    eval_set = ev_all,
    verbose = 0
)

yhat_final = clf_final.predict(xtest)

clf_final.save_model("model-exports/aktuelle-exports/fs_embedded_harus.json")
clf_final.get_booster().dump_model("model-exports/aktuelle-exports/fs_embedded_harus_dump.json")

### 2.1.9 Evaluierung

In [ ]:
# Konfusionsmatrizen plotten

skm.ConfusionMatrixDisplay.from_estimator(clf_final, xtest, ytest, cmap='Blues', normalize='true', values_format='.2f')
plt.show()

skm.ConfusionMatrixDisplay.from_estimator(clf_final, xtest, ytest, cmap="Blues" ,values_format='d')
plt.show()

print(skm.classification_report(ytest, yhat_final, labels=[0,1], target_names=['0: standing','1: walking'], digits=6))

# Kennzahlen ermitteln

tp, fp, tn, fn = 4621,0,5761,0
acc = skm.accuracy_score(ytest, yhat_final)
sens = tp/(tp + fn)
spec = tn/(tn + fn)
prec = tp/(tp + fp)
npv = tn/(tn + fn)
fscore = 2 * ((prec * sens) / (prec + sens))

print("DIE GANZEN ERGEBNISSE HIER STIMMEN NICHT/GEHOEREN ZU SEMU!!!!!!!!!!!!!")
print(f"Genauigkeit: \t\t {'%.6f' % acc}")
print(f"Präzision: \t\t {'%.6f' % prec}")
print(f"Sensitivität: \t\t {'%.6f' % sens}")
print(f"Spezifität: \t\t {'%.6f' % spec}")
print(f"NPR: \t\t\t {'%.6f' % npv}")
print(f"F-Maß: \t\t\t {'%.6f' % fscore}\n")

print(f"Maximale Baumtiefe: \t {best_depth}")
print(f"Anzahl Schätzer: \t {bestIterList[best_depth-1]}")
print(f"Anzahl der Knoten: \t MUSS ICH NOCH HERAUSFINDEN.....")


## 2.2 Wrapper-Methode: Rekursive Feature Eliminierung (CV)

In [ ]:
min_features_to_select = 1  # Minimum number of features to consider

clf = XGBClassifier(
    objective = "multi:softmax",
    tree_method = "exact"
)
cv = StratifiedKFold(5)

rfecv = RFECV(
    estimator=clf,
    step=1,
    cv=cv,
    scoring="accuracy",
    min_features_to_select=min_features_to_select,
    n_jobs=-1,
)
rfecv.fit(xtrain_old, ytrain)

print(f"Optimal number of features: {rfecv.n_features_}")

cv_results = pd.DataFrame(rfecv.cv_results_)
plt.figure()
plt.xlabel("Number of features selected")
plt.ylabel("Mean test accuracy")
plt.errorbar(
    x=cv_results["n_features"],
    y=cv_results["mean_test_score"],
    yerr=cv_results["std_test_score"],
)
# plt.xlim(65,80)
plt.title("Recursive Feature Elimination \nwith correlated features")
plt.show()

In [ ]:
xtrain_filtered_list = [col for col, keep in zip(xtrain_old.columns, rfecv.support_) if keep]

xtrain_rfecv = xtrain_old[xtrain_filtered_list]
xval_rfecv = xval_old[xtrain_filtered_list]
xtest_rfecv = xtest_old[xtrain_filtered_list]
